In [ ]:
import os, sys
from pathlib import Path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import pandas as pd
import importlib

import geoquant.configs.config as config
import geoquant.data_io as f1
import geoquant.series_utils as f2
import geoquant.portfolio as portfolio
import geoquant.risk_matrix as risk_matrix
import geoquant.books as books

importlib.reload(books)
importlib.reload(risk_matrix)
importlib.reload(f1)
importlib.reload(f2)

data_params = dict(config.data_params)
holdings = books.IBKR_live

print('analysis window:', data_params.get('start'), 'to', data_params.get('end'))
print('max age:', data_params.get('max_age'))
print('---- Building returns matrix in CHF ----')
rets_df, prices_df, weights, rebalance_df = risk_matrix.build_returns_weights(
    holdings,
    data_params=data_params,
    no_fx=False,
    usd_shift=True,
    use_target_weights=True,
    include_cash=False,
    plot_spikes=False,
 )

print('weights\n', weights)
if rebalance_df is not None:
    print('\nrebalancing:\n', rebalance_df[['target_weight','current_chf','desired_chf','delta_chf','current_position','desired_position','delta_shares']].round({'target_weight':3,'current_chf':0,'desired_chf':0,'delta_chf':0,'current_position':0,'desired_position':1,'delta_shares':1}))

print('---- Calculating portfolio risk ----')
results = risk_matrix.portfolio_risk(rets_df, weights)

pf_vol = results['port_vol']
summary = results['summary'].copy()

print('summary\n', summary)
assets = summary.index
w = summary['Weight'].astype(float)
mrc = summary['MRC'].astype(float)

desired_dd = 0.04

ohlc = True
fx_map = portfolio.make_fx_map(holdings, data_params, usd_shift=True, ohlc=ohlc)
high_df, low_df, close_df, close_local_df = portfolio.base_ccy_assets_px_df(holdings, fx_map, data_params, ohlc)

# 1) COMPUTE ATR% (WILDER) FROM OHLC IN THE SAME CURRENCY BASIS AS WEIGHTS/RETURNS (CHF)
high = high_df[assets]
low = low_df[assets]
close = close_df[assets]

prev_close = close.shift(1)
tr = pd.concat([
    (high - low),
    (high - prev_close).abs(),
    (low - prev_close).abs()
], axis=1).groupby(level=0, axis=1).max()

atr = tr.ewm(alpha=1/14, adjust=False).mean()
atr_pct = (atr.iloc[-1] / close.iloc[-1]).astype(float).clip(lower=1e-6)
atr_pct = atr_pct.reindex(assets)
assert atr_pct.index.equals(assets)

# 2) PER-ASSET DD BUDGETS BY ABSOLUTE TRC
trc_abs = (w * mrc).abs()
share = trc_abs / trc_abs.sum()
b = desired_dd * share

# 3) L1 STOP DISTANCES AND ATR MULTIPLES
abs_w = w.abs().clip(lower=1e-12)

cat = {
    'EMIM': 'core',
    'VUAG': 'core',
    'VEU': 'core',
    'ISJP': 'core',
    'IEMS': 'core',
    'XXSC': 'core',
    'IPLT': 'diversifier',
    'REMX': 'diversifier',
    'INRG': 'diversifier',
    'SGLN': 'diversifier',
}

cat_s = pd.Series({a: cat.get(a, 'tactical') for a in assets}).reindex(assets)
defaults = [a for a in assets if a not in cat]
print('Asset categories:\n', cat_s)
print("Defaulted to 'tactical':", defaults)
print(cat_s.value_counts())

min_k_by_cat = {'core': 4.0, 'tactical': 2.0, 'diversifier': 5.0, 'hedge': float('inf')}
include_in_stops = {'core': True, 'tactical': True, 'diversifier': True, 'hedge': False}

min_k = cat_s.map(min_k_by_cat).astype(float)
take_stop = cat_s.map(include_in_stops).fillna(True).astype(bool)

min_pct, max_pct = 0.002, 0.10
s = (b / abs_w).astype(float)
floor = ((min_k * atr_pct).astype(float)).clip(lower=min_pct)
s.loc[~take_stop] = np.nan
s.loc[take_stop] = np.minimum(np.maximum(s.loc[take_stop].values, floor.loc[take_stop].values), max_pct)

def _L1(total):
    return float((abs_w[take_stop] * total[take_stop]).sum())

L1_floor = _L1(floor)
if L1_floor > desired_dd + 1e-10:
    print(f'Infeasible: floors imply min DD {L1_floor:.4f} > target {desired_dd:.4f}. Using floors.')
    s.loc[take_stop] = floor.loc[take_stop]
else:
    def enforce_L1_with_floors(abs_w, s, floor, take_stop, dd, tol=1e-10, max_iter=20):
        s = s.copy()
        for _ in range(max_iter):
            L1 = float((abs_w[take_stop] * s[take_stop]).sum())
            if L1 <= dd + tol:
                break
            free = take_stop & (s > floor + 1e-12)
            if not free.any():
                break
            W = float(abs_w[free].sum())
            if W <= 0:
                break
            delta = (L1 - dd) / max(W, 1e-12)
            s.loc[free] = np.maximum(floor[free], s[free] - delta)
        return s

    s = enforce_L1_with_floors(abs_w, s, floor, take_stop, desired_dd)

# Final stops and diagnostics
stop_pct = s.astype(float)
k_multiple = (stop_pct / atr_pct).astype(float)
dd_check = float((abs_w[take_stop] * stop_pct[take_stop]).sum())
print(f'L1 sum check ~ {dd_check:.4f} (target {desired_dd:.4f})')
binds = (take_stop & (stop_pct <= floor + 1e-12))
if binds.any():
    print('At floor for:', list(stop_pct[binds].index))

# 4) STOP PRICES
last_px_local = close_local_df[assets].iloc[-1].astype(float)
stop_px_local = pd.Series(
    np.where(w >= 0, last_px_local * (1.0 - stop_pct), last_px_local * (1.0 + stop_pct)),
    index=assets, name='stop_price_local'
 )
last_px_chf = close.iloc[-1].astype(float).reindex(assets)
stop_px_chf = pd.Series(
    np.where(w >= 0, last_px_chf * (1.0 - stop_pct), last_px_chf * (1.0 + stop_pct)),
    index=assets, name='stop_price'
 )

out = pd.DataFrame({
    'Weight': w,
    'ATR_pct': round(atr_pct * 100, 2),
    'stop_pct': round(stop_pct * 100, 2),
    'k_ATR_mult': k_multiple,
    'last_px_local': last_px_local,
    'stop_px_local': stop_px_local,
    'last_px_chf': last_px_chf,
    'stop_px_chf': stop_px_chf
}).round(6)

print('Portfolio σ (annualized, CHF):', pf_vol)
print(out)

max age: 24
---- Building returns matrix in CHF ----


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
# Deprecated scratch cell kept intentionally empty.
# Main vol-stop workflow is in Cell 1.